In [9]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [10]:
def generate_data(num_samples, seq_len):
    x = np.random.randint(0, 10, size=(num_samples, seq_len))
    y = np.zeros_like(x)

    y[:, 0] = x[:, 0]
    for i in range(1, seq_len):
        y[:, i] = (x[:, i] + x[:, 0]) % 10

    return x, y


In [11]:
SEQ_LEN = 15
NUM_SAMPLES_TRAIN = 5000
NUM_SAMPLES_TEST = 500
BATCH_SIZE = 64

x_train, y_train = generate_data(NUM_SAMPLES_TRAIN, SEQ_LEN)
x_test, y_test = generate_data(NUM_SAMPLES_TEST, SEQ_LEN)

def to_one_hot(x, num_classes=10):
    return np.eye(num_classes)[x]

x_train_oh = to_one_hot(x_train)
x_test_oh = to_one_hot(x_test)

X_train_tensor = torch.FloatTensor(x_train_oh)
Y_train_tensor = torch.LongTensor(y_train)

X_test_tensor = torch.FloatTensor(x_test_oh)
Y_test_tensor = torch.LongTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [12]:
# кдасс модели
class SequencePredictor(nn.Module):
    def __init__(self, model_type='LSTM', hidden_size=32, num_layers=1):
        super(SequencePredictor, self).__init__()
        self.model_type = model_type

        if model_type == 'RNN':
            self.rnn = nn.RNN(input_size=10, hidden_size=hidden_size,
                              num_layers=num_layers, batch_first=True)
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(input_size=10, hidden_size=hidden_size,
                               num_layers=num_layers, batch_first=True)
        elif model_type == 'GRU':
            self.rnn = nn.GRU(input_size=10, hidden_size=hidden_size,
                              num_layers=num_layers, batch_first=True)
        else:
            raise ValueError("Выберите RNN, LSTM или GRU.")

        self.fc = nn.Linear(hidden_size, 10)

    def forward(self, x):
        out, _ = self.rnn(x)
        logits = self.fc(out)
        return logits

In [13]:
# запуск и трейн
def train_and_evaluate(model_type):
    print(f"\n Обучается модель: {model_type}")

    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    model = SequencePredictor(model_type=model_type, hidden_size=64).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    epochs = 15
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_x)

            loss = criterion(logits.view(-1, 10), batch_y.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader)}")

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_tensor.to(device))
        predictions = torch.argmax(test_logits, dim=-1).cpu().numpy()

    accuracy = np.mean(predictions == y_test)
    print(f"Точность на тестовой выборке: {accuracy * 100}%")

    return accuracy

In [14]:
torch.manual_seed(42)
np.random.seed(42)

acc_rnn = train_and_evaluate('RNN')
acc_lstm = train_and_evaluate('LSTM')
acc_gru = train_and_evaluate('GRU')

print(f"RNN  Accuracy: {acc_rnn * 100}%")
print(f"LSTM Accuracy: {acc_lstm * 100}%")
print(f"GRU  Accuracy: {acc_gru * 100}%")


 Обучается модель: RNN
Epoch 5/15, Loss: 2.2921460882017883
Epoch 10/15, Loss: 2.1701906240439115
Epoch 15/15, Loss: 1.838511785374412
Точность на тестовой выборке: 26.813333333333333%

 Обучается модель: LSTM
Epoch 5/15, Loss: 0.006316861892236939
Epoch 10/15, Loss: 0.0012085984381434473
Epoch 15/15, Loss: 0.0005288174434099346
Точность на тестовой выборке: 100.0%

 Обучается модель: GRU
Epoch 5/15, Loss: 0.018364293878025646
Epoch 10/15, Loss: 0.002723467836887399
Epoch 15/15, Loss: 0.0011802557841108382
Точность на тестовой выборке: 100.0%
RNN  Accuracy: 26.813333333333333%
LSTM Accuracy: 100.0%
GRU  Accuracy: 100.0%


RNN показала наихудший результат, так как у нее отсутствуют механизмы управления памятью и затухающего градиента. Начальная информация теряется к концу последовательности.

GRU и LTSM напротив показали хороший результат точности, так как могут сохранять x1 в памяти и использовать на каждом шаге обучения.